In [105]:
import pandas as pd

countryC = pd.read_csv(
    "https://raw.githubusercontent.com/Imagin-io/country-nationality-list/master/countries.csv"
)

countryC = countryC.rename(
    columns={"alpha_3_code": "country_code", "nationality": "nationality"}
)

countryC = countryC[["country_code", "nationality"]].drop_duplicates()

# Add missing Formula 1 nationality codes
extra_countries = pd.DataFrame({
    "country_code": [
        "DEN", "MON", "NED", "GER", "RSA", "SUI",
        "GBR", "POR", "URS", "ROC", "BUL"
    ],
    "nationality": [
        "Danish", "Monegasque", "Dutch", "German",
        "South African", "Swiss", "British", "Portuguese",
        "Soviet", "Russian", "Bulgarian"
    ]
})

# Add only codes not already present
countryC = pd.concat(
    [
        countryC,
        extra_countries[
            ~extra_countries["country_code"].isin(countryC["country_code"])
        ]
    ],
    ignore_index=True
)

# Remove missing values and duplicates
countryC = countryC.dropna(
    subset=["country_code", "nationality"]
).drop_duplicates(subset=["country_code"])

with open("cleaned/Country_Codes.csv", "w") as f:
    countryC.to_csv(f, index=False)

In [112]:
import pandas as pd

drivers = pd.read_csv("formula_one_driver.csv").drop(
    columns=["abbreviation", "api_id", "permanent_car_number", "reference", "wikipedia"]
).rename(columns={"id": "driver_id"})

df = pd.read_csv("formula_one_teamdriver.csv").drop(columns=["api_id", "role"])
df = df.drop(columns=["id"])

teams = pd.read_csv("formula_one_team.csv").drop(
    columns=["base_team_id", "api_id", "primary_color", "reference", "wikipedia"]
).rename(columns={"id": "team_id"})

season = pd.read_csv("formula_one_season.csv").drop(
    columns=["api_id", "wikipedia"]
).rename(columns={"id": "season_id"})


drivers = drivers.rename(columns={
    col: f"drivers_{col}" for col in drivers.columns
    if col != "driver_id"
})

cols = [
    "drivers_forename",
    "drivers_surname",
    "drivers_date_of_birth",
    "drivers_nationality",
    "drivers_country_code",
    "driver_id",
]

# Option 1: Reassign directly (recommended)
drivers = drivers[cols]

# fill drivers with missing data
# birthdate
missingBds = {
  "864": "1999-05-06",
  "865": "2001-09-22",
  "866": "2000-10-14",
  "867": "2002-01-31",
  "868": "2004-02-04",
  "869": "2005-05-02",
  "870": "2005-03-07",
  "873": "2004-01-19",
  "874": "2005-11-11",
  "875": "2001-06-16",
  "876": "2002-01-13",
  "877": "1996-05-20",
  "878": "2000-05-23",
  "879": "2000-03-30",
  "880": "2004-12-03",
  "881": "2008-08-09",
  "882": "2006-12-21",
  "883": "2005-03-20"
}

# fill the missing birthdates in the drivers dataframe
for idx, birthdate in missingBds.items():
    drivers.loc[int(idx), "drivers_date_of_birth"] = birthdate

def find_col(df, substring):
    matches = [c for c in df.columns if substring in c]
    if not matches:
        raise KeyError(f"Nessuna colonna contenente '{substring}' in {list(df.columns)}")
    return matches[0]


drv_code_col = find_col(drivers, "country_code")
drv_nat_col = find_col(drivers, "nationality")
cnt_code_col = find_col(countryC, "country_code")
cnt_nat_col = find_col(countryC, "nationality")

# Mappa {country_code: nationality} da countryC (senza duplicati)
nat_map = (
    countryC[[cnt_code_col, cnt_nat_col]]
    .dropna(subset=[cnt_code_col, cnt_nat_col])
    .drop_duplicates(subset=[cnt_code_col])
    .set_index(cnt_code_col)[cnt_nat_col]
    .to_dict()
)

# Aggiorna solo i driver a cui a manca la nazionalità ma hanno un country_code valido
target_ids = drivers.loc[drivers[drv_nat_col].isna() & drivers[drv_code_col].notna(), "driver_id"]
mask = drivers["driver_id"].isin(target_ids)

# Riallinea i country_code mancanti se necessario e poi mappa la nazionalità
drivers.loc[mask, drv_nat_col] = (
    drivers.loc[mask, drv_code_col]
    .map(nat_map)
    .fillna(drivers.loc[mask, drv_nat_col])
)


teams = teams.rename(columns={
    col: f"teams_{col}" for col in teams.columns
    if col != "team_id"
})

season = season.rename(columns={
    col: f"season_{col}" for col in season.columns
    if col != "season_id"
})

season = season[["season_year", "season_id", "season_championship_system_id"]]

teams = teams[["teams_name", "teams_nationality", "teams_country_code", "team_id"]]


with open("cleaned/Drivers_cleaned.csv", "w") as f:
    drivers.to_csv(f, index=False)

with open("cleaned/Teams_cleaned.csv", "w") as f:
    teams.to_csv(f, index=False)

with open("cleaned/Season_cleaned.csv", "w") as f:
    season.to_csv(f, index=False)


df = pd.merge(df, drivers, on="driver_id", how="left")
df = pd.merge(df, teams, on="team_id", how="left")
df = pd.merge(df, season, on="season_id", how="left")

ids = [df.columns[x] for x in range(len(df.columns)) if df.columns[x].endswith("id")]
ids2 = [df.columns[x] for x in range(len(df.columns)) if df.columns[x] not in ids]

df = df[ids2 + ids]

with open("cleaned/Merged_drivers_info.csv", "w") as f:
    df.to_csv(f, index=False)

In [114]:
def checkforNAN(df):
    missing = df.isna() | df.astype("string").apply(
        lambda col: col.str.strip().eq("")
    )

    missing_cols = missing.columns[missing.any()].tolist()

    if missing_cols:
        print("columns with missing values:")
        for col in missing_cols:
            print(f"{col}: {missing[col].sum()} missing values")
    else:
        print("No missing values.")